# Stopped-Process Expectation Generator

Notebook workbench for generating prompts, reasoning traces, and canonical answers for stopped-process expectation problems.

In [1]:
from pathlib import Path
import json
import sys

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "benchmark").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
from benchmark.generators import StoppedProcessExpectationGenerator

gen = StoppedProcessExpectationGenerator()

In [3]:
params = gen.sample_params(seed=301, difficulty=2, split="dev")
params

{'problem_type': 'quadratic_fixed_horizon',
 'difficulty': 2,
 'split': 'dev',
 'seed': 301,
 'process_notation': 'X',
 'start': 2,
 'horizon': 7,
 'step': 5,
 'variance': 25,
 'value': Fraction(179, 1)}

In [4]:
problem = gen.generate_problem(params)
reasoning = gen.generate_reasoning(params)
solution = gen.generate_solution(params)

print("PROBLEM:\n", problem)
print("\nREASONING:\n", reasoning)
print("\nSOLUTION:\n", gen.to_json_safe(solution))

PROBLEM:
 Let X_n = 2 + Y_1 + ... + Y_n, where P(Y_k = 5) = P(Y_k = -5) = 1/2 and the increments are independent. Compute E[X_7^2]. Answer with JSON of the form {"value": "..."} inside the answer tags.

REASONING:
 Use the quadratic martingale for centered independent increments: if S_n = S_0 + Y_1 + ... + Y_n, the increments have mean 0 and variance sigma^2, and the increments are independent of the past, then M_n = S_n^2 - n sigma^2 is a martingale. Since Var(Y_k) = 25, M_n = X_n^2 - 25 n is a martingale with M_0 = 2^2. Taking expectations at the deterministic time 7 gives E[X_7^2] - 25 * 7 = 2^2, so E[X_7^2] = 2^2 + 7 * 25.

Final answer:
<answer>
{"value": "179"}
</answer>

SOLUTION:
 {'value': '179'}


In [5]:
records = []
for difficulty in [1, 2, 3]:
    for seed in range(3000 + 100 * difficulty, 3005 + 100 * difficulty):
        records.append(gen.generate_record(seed=seed, difficulty=difficulty, split="dev"))

len(records), records[0]

(15,
 {'id': 'stopped_process_expectation_dev_003100',
  'family': 'stopped_process_expectation',
  'problem_type': 'bounded_walk_expectation',
  'difficulty': 1,
  'split': 'dev',
  'seed': 3100,
  'params': {'problem_type': 'bounded_walk_expectation',
   'difficulty': 1,
   'split': 'dev',
   'seed': 3100,
   'process_notation': 'X',
   'start': 2,
   'horizon': 5,
   'level': 4,
   'value': '2'},
  'problem': 'Let (X_n) be a simple symmetric random walk with X_0 = 2. Let T = inf{k >= 0 : X_k = 4} and tau = min(T, 5). Compute E[X_tau]. Answer with JSON of the form {"value": "..."} inside the answer tags.',
  'reasoning': 'Use the bounded optional stopping theorem: if (M_n) is a martingale and tau is a bounded stopping time, then E[M_tau] = E[M_0]. The process (X_n) is a martingale, and tau is bounded by the deterministic horizon. Therefore E[X_tau] = E[X_0] = 2.\n\nFinal answer:\n<answer>\n{"value": "2"}\n</answer>',
  'canonical_answer': {'value': '2'},
  'metadata': {'theorem_label

In [6]:
output_path = project_root / "benchmark" / "data" / "dev" / "stopped_process_expectation_preview.jsonl"
with output_path.open("w") as f:
    for record in records:
        f.write(json.dumps(record, sort_keys=True) + "\n")

output_path

PosixPath('/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/benchmark/data/dev/stopped_process_expectation_preview.jsonl')